# Config

Shared setup for every stage notebook, across all three pipelines (employees,
sales, inventory). Run this first with `%run ../00_config.ipynb`.

Bootstraps the Kedro project so `catalog` (the project's `DataCatalog`) is available for `catalog.load(...)` / `catalog.save(...)` in every stage notebook -- reads and writes always go through the catalog, not hardcoded paths.

In [ ]:
from pathlib import Path

from kedro.framework.startup import bootstrap_project
from kedro.framework.session import KedroSession


def _find_project_root(start: Path) -> Path:
    path = start.resolve()
    while not (path / "pyproject.toml").exists():
        if path == path.parent:
            raise FileNotFoundError("Could not locate project root (pyproject.toml)")
        path = path.parent
    return path


PROJECT_ROOT = _find_project_root(Path.cwd())

bootstrap_project(PROJECT_ROOT)
session = KedroSession.create(PROJECT_ROOT)
context = session.load_context()
catalog = context.catalog

EMPLOYEES_GOLD_COLUMNS = ["employee_id", "full_name", "department", "salary", "hire_date"]
SALES_GOLD_COLUMNS = ["order_id", "customer_name", "product", "quantity", "unit_price", "order_date"]
INVENTORY_GOLD_COLUMNS = ["item_id", "item_name", "category", "stock_count", "warehouse", "last_updated"]

print(f"Kedro context loaded from {PROJECT_ROOT}")
print("Catalog datasets:", list(catalog.keys()))

## PySpark alternative (reference only)

PySpark isn't installed in this image. This is left commented out to show
how the session bootstrap would look if the project were switched to Spark.

In [ ]:
# from pyspark.sql import SparkSession
#
# spark = (
#     SparkSession.builder
#     .appName("kedro_oozie_demo")
#     .master("local[*]")
#     .getOrCreate()
# )